In [1]:
!pip install rdkit
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, MACCSkeys, AllChem
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from imblearn.over_sampling import SMOTE
import joblib
import warnings
warnings.filterwarnings('ignore')




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 52.3 MB/s eta 0:00:00


In [2]:
df = pd.read_csv('/content/train.csv')
df.shape

(7697, 3)

In [3]:
def validate_smiles(smiles):
    """Convert SMILES to RDKit Mol, return None if invalid"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    try:
        Chem.SanitizeMol(mol)
        return mol
    except:
        return None

# Apply validation
df['mol'] = df['SMILES'].apply(validate_smiles)

# Remove invalid molecules
df = df.dropna(subset=['mol']).reset_index(drop=True)

print(f"\n Valid training molecules: {len(df):,}")

[05:39:49] Explicit valence for atom # 2 Cl, 1, is greater than permitted
[05:39:50] Explicit valence for atom # 0 Cl, 1, is greater than permitted
[05:39:50] WARNING: not removing hydrogen atom without neighbors



 Valid training molecules: 7,695


In [4]:
def generate_toxicity_features(mol):
    morgan_fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
    maccs_fp = MACCSkeys.GenMACCSKeys(mol)

    # B. PHYSICOCHEMICAL DESCRIPTORS (ADME/toxicity factors)
    descriptors = np.array([
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.NumRotatableBonds(mol),
        rdMolDescriptors.CalcNumRings(mol),
        Descriptors.FractionCSP3(mol),
        rdMolDescriptors.CalcNumAromaticRings(mol),
        Descriptors.NumAromaticRings(mol)
    ])

    # Combine: [Morgan 2048] + [MACCS 166] + [Descriptors 10] = 2224 features
    features = np.concatenate([np.array(morgan_fp), np.array(maccs_fp), descriptors])
    return features

# Generate features for train/test
X = np.array([generate_toxicity_features(mol) for mol in df['mol']])
#y = df['Label'].values


Streaming output truncated to the last 5000 lines.
[05:40:04] DEPRECATION WARNING: please use MorganGenerator
[05:40:04] DEPRECATION WARNING: please use MorganGenerator
[05:40:04] DEPRECATION WARNING: please use MorganGenerator
[05:40:04] DEPRECATION WARNING: please use MorganGenerator
[05:40:04] DEPRECATION WARNING: please use MorganGenerator
[05:40:04] DEPRECATION WARNING: please use MorganGenerator
[05:40:04] DEPRECATION WARNING: please use MorganGenerator
[05:40:04] DEPRECATION WARNING: please use MorganGenerator
[05:40:04] DEPRECATION WARNING: please use MorganGenerator
[05:40:04] DEPRECATION WARNING: please use MorganGenerator
[05:40:04] DEPRECATION WARNING: please use MorganGenerator
[05:40:04] DEPRECATION WARNING: please use MorganGenerator
[05:40:04] DEPRECATION WARNING: please use MorganGenerator
[05:40:04] DEPRECATION WARNING: please use MorganGenerator
[05:40:04] DEPRECATION WARNING: please use MorganGenerator
[05:40:04] DEPRECATION WARNING: please use MorganGenerator
[05:4

In [5]:

from sklearn.metrics import accuracy_score
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

In [10]:
from joblib import dump, load
# Later, to load it back
loaded_model = load("/content/model_13_B2.pt")

In [11]:
X_train, X_val, y_train, y_val = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)
y_val_pred = loaded_model.predict(X_val)

In [13]:
import pandas as pd

# Suppose you want to save predictions along with their index
df = pd.DataFrame({
    "Index": range(len(y_val_pred)),
    "Prediction": y_val_pred
})

# Save to CSV
df.to_csv("Submission_13_B2.csv", index=False)
